In [74]:
import re
import glob
import zipfile
import time
from pathlib import Path
import json
import requests
import pandas as pd
from urllib.request import urlretrieve

In [75]:
import time, psutil, os, functools

_proc = psutil.Process(os.getpid())

def measure(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        m0 = _proc.memory_info().rss / 1e6
        c0 = time.process_time()
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        print(f"{fn.__name__}:  wall={time.perf_counter()-t0:.1f}s  "
              f"cpu={time.process_time()-c0:.1f}s  "
              f"mem Δ{_proc.memory_info().rss/1e6 - m0:+.0f} MB")
        return out
    return wrapper

In [76]:
WORKDIR = Path.cwd()
OUTPUT_DIR = WORKDIR / "data" ## Change it to the location that you want to download your files to.
OUTPUT_DIR.mkdir(exist_ok=True)
FORCE_DOWNLOAD = False 

In [77]:
article_id = 14096681  # this is the unique identifier of the article on figshare
url = f"https://api.figshare.com/v2/articles/{article_id}"
headers = {"Content-Type": "application/json"}
output_directory = "data/"

In [78]:
response = requests.request("GET", url, headers=headers)
data = json.loads(response.text)  # this contains all the articles data, feel free to check it out
files = data["files"]             # this is just the data about the files, which is what we want
files

[{'id': 26579150,
  'name': 'daily_rainfall_2014.png',
  'size': 58863,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579150',
  'supplied_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'computed_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'mimetype': 'image/png'},
 {'id': 26579171,
  'name': 'environment.yml',
  'size': 192,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579171',
  'supplied_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'computed_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'mimetype': 'text/plain'},
 {'id': 26586554,
  'name': 'README.md',
  'size': 5422,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26586554',
  'supplied_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'computed_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'mimetype': 'text/x-python'},
 {'id': 26766812,
  'name': 'data.zip',
  'size': 814041183,
  'is_link_only': False,
  'download_url': 'https://n

In [79]:
files_to_dl = ["data.zip"]  # feel free to add other files here
for file in files:
    if file["name"] in files_to_dl:

        output_path = OUTPUT_DIR / file["name"]

        if output_path.exists() and not FORCE_DOWNLOAD:
            print(f"{file['name']} already exists. Skipping.")
        else:
            output_path.parent.mkdir(exist_ok=True)  # ensure folder exists

            print(f"Downloading {file['name']}...")
            urlretrieve(file["download_url"], output_path)

data.zip already exists. Skipping.


In [80]:
zip_path = OUTPUT_DIR / "data.zip"
extract_dir = OUTPUT_DIR / "extracted"

if not extract_dir.exists():
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as f:
        f.extractall(extract_dir)
else:
    print("Files already extracted. Skipping extraction.")

Files already extracted. Skipping extraction.


In [81]:
### just listing to get an idea how individual file looks like 
use_cols = ["time", "lat_min", "lat_max", "lon_min", "lon_max", "rain (mm/day)"]


In [82]:

@measure
def combine_csvs(output_path):
    FORCE_MERGE = False
    if output_path.exists() and not FORCE_MERGE:
        print("Merged file already exists. Skipping.")
    else:
        files = glob.glob("data/extracted/*.csv")

        df = pd.concat(
            (pd.read_csv(file, index_col=0, usecols=use_cols)
            .assign(model=Path(file).stem.split("_")[0]) #Modified this line, as regex does not work great for my directory structure/organization
            for file in files if "observed_daily_rainfall_SYD.csv" not in file)
        )

        df.to_csv(output_path)

output_path = Path("data/combined_data.csv")
combine_csvs(output_path)

combine_csvs:  wall=156.3s  cpu=155.6s  mem Δ+2728 MB


ANSWER:

From the table below, there are noticeable differences in wall and CPU time, with some machines performing faster than others. The wall time is very close to the CPU time, which suggests the task is mostly CPU-bound rather than waiting on disk I/O. The wall/CPU ratio helps determine whether a task is CPU-bound or I/O-bound: if the ratio is close to 1, the task is CPU-bound, while a higher ratio indicates more time spent waiting on I/O. This means most of the time is spent processing the data instead of reading it. One surprising observation is that machines with similar specs can still have different performance, likely due to system differences or background processes.

In [83]:
@measure
def column_load():
    use_cols = ["model", "lat_min", "lat_max"]
    df = pd.read_csv("data/combined_data.csv",usecols=use_cols)
    print(df["model"].value_counts())

column_load()

model
MPI-ESM1-2-HR       5154240
CMCC-CM2-HR4        3541230
CMCC-ESM2           3541230
TaiESM1             3541230
NorESM2-MM          3541230
CMCC-CM2-SR5        3541230
SAM0-UNICON         3541153
FGOALS-f3-L         3219300
GFDL-CM4            3219300
GFDL-ESM4           3219300
MRI-ESM2-0          3037320
EC-Earth3-Veg-LR    3037320
BCC-CSM2-MR         3035340
MIROC6              2070900
ACCESS-CM2          1932840
ACCESS-ESM1-5       1610700
INM-CM4-8           1609650
INM-CM5-0           1609650
FGOALS-g3           1287720
KIOST-ESM           1287720
MPI-ESM-1-2-HAM      966420
AWI-ESM-1-1-LR       966420
NESM3                966420
MPI-ESM1-2-LR        966420
NorESM2-LM           919800
BCC-ESM1             551880
CanESM5              551880
Name: count, dtype: int64
column_load:  wall=13.3s  cpu=13.3s  mem Δ-1648 MB


In [84]:
@measure
def data_type_load():
    df = pd.read_csv("data/combined_data.csv",
                     parse_dates=["time"],
                     dtype={
                          "lat_min": "float32",
                          "lat_max": "float32",
                          "lon_min": "float32",
                          "lon_max": "float32",
                          "rain (mm/day)": "float32",
                          "model": "category"
                     }
                     )
    print(df["model"].value_counts())

data_type_load()

model
MPI-ESM1-2-HR       5154240
CMCC-CM2-HR4        3541230
CMCC-ESM2           3541230
TaiESM1             3541230
NorESM2-MM          3541230
CMCC-CM2-SR5        3541230
SAM0-UNICON         3541153
FGOALS-f3-L         3219300
GFDL-CM4            3219300
GFDL-ESM4           3219300
MRI-ESM2-0          3037320
EC-Earth3-Veg-LR    3037320
BCC-CSM2-MR         3035340
MIROC6              2070900
ACCESS-CM2          1932840
ACCESS-ESM1-5       1610700
INM-CM4-8           1609650
INM-CM5-0           1609650
FGOALS-g3           1287720
KIOST-ESM           1287720
MPI-ESM-1-2-HAM      966420
AWI-ESM-1-1-LR       966420
NESM3                966420
MPI-ESM1-2-LR        966420
NorESM2-LM           919800
BCC-ESM1             551880
CanESM5              551880
Name: count, dtype: int64
data_type_load:  wall=24.4s  cpu=24.3s  mem Δ+1900 MB


ANSWER:

Based on my results, the usecols approach was more effective in improving performance. It significantly reduced both wall and CPU time by limiting the amount of columns to read. In contrast, the dtype approach reduced memory usage, even resulting in a negative memory delta, but did not improve runtime and was slower due to additional processing overhead. During the lab, I asked whether we should use the same set of columns for a fair comparison, but both the TA and instructor indicated this was not required. As a result, the comparison is not strictly fair, since usecols reduces the amount of data read while dtype operates on the full dataset. In practice, I would choose usecols when only a subset of columns is needed for better performance, and use dtype when the full dataset must be loaded but memory efficiency is important.

In [85]:
import duckdb

@measure
def duckdb_eda():
    return duckdb.sql("""
        SELECT
            model,
            COUNT(*) AS n
        FROM read_csv_auto('data/combined_data.csv')
        GROUP BY model
        ORDER BY n DESC
    """).df()

result = duckdb_eda()
result

duckdb_eda:  wall=1.1s  cpu=10.1s  mem Δ+151 MB


,model,n
0,MPI-ESM1-2-HR,5154240
1,CMCC-ESM2,3541230
2,NorESM2-MM,3541230
3,CMCC-CM2-HR4,3541230
4,CMCC-CM2-SR5,3541230
5,TaiESM1,3541230
6,SAM0-UNICON,3541153
7,GFDL-CM4,3219300
8,FGOALS-f3-L,3219300
9,GFDL-ESM4,3219300


Compared to my usecols pandas approach, DuckDB has a much smaller memory delta because it processes the data directly from the CSV without loading everything into memory. The wall time is much lower than CPU time, which suggests efficient execution with parallel processing. I would prefer DuckDB for large datasets when performing simple aggregations while minimizing memory usage.

In [86]:
import os

base_path = "data"

filepathcsv = f"{base_path}/combined_data.csv"
filepathparquet = f"{base_path}/combined_data.parquet"
filepathparquetr = f"{base_path}/combined_data_r.parquet"

os.environ['R_HOME'] = '/Users/gittugeorge/miniforge3/envs/525_dev_2026_latest/lib/R'

In [87]:

import pyarrow.dataset as ds
df = ds.dataset(filepathcsv, format="csv")
result = df.scanner(columns=["time", "lat_min", "lat_max", "lon_min", "lon_max", "rain (mm/day)", "model"])
ds.write_dataset(result,filepathparquet,format = "parquet",partitioning=["model"],partitioning_flavor="hive", existing_data_behavior = 'overwrite_or_ignore')

In [88]:
# ```{r}
# library(arrow)
# library(dplyr)

# base_path <- "data"

# filepathcsv <- file.path(base_path, "combined_data.csv")
# filepathparquet <- file.path(base_path, "combined_data.parquet")
# filepathparquetr <- file.path(base_path, "combined_data_r.parquet")


# filetest <- arrow::open_dataset(filepathcsv, format="csv") |>
#   select("time", "lat_min", "lat_max", "lon_min", "lon_max", "rain (mm/day)", "model")

# eda <- filetest |>
#   count(model, sort = TRUE) |>
#   collect()

# eda
# ```

I chose the Parquet file approach because it is easy to use and works well between Python and R. Instead of setting up more complex tools like pandas exchange or Arrow, I can just save the dataset once and load it in R without extra configuration. This also avoids issues with dependencies like rpy2. Additionally, Parquet is efficient for large datasets, which makes it a practical choice for this milestone.

| Member  | Approach     | OS      | RAM  | Processor     | SSD? | wall (s) | cpu (s) | mem Δ (MB) |
|---------|-------------|---------|------|---------------|------|----------|---------|------------|
| Eduardo | combine     | macOS   | 48GB | Apple M4 Pro  | Y    | 152.6    | 152.2   | +2149      |
| Suryash | combine     | windows | 32GB | Intel Core i9 | Y    | 44.6     | 44.0    | +3302      |
| Rabin   | combine     | macOS     | 24GB | M4 Pro        | Y    | 146.4    | 145.6   | +6164      |
| Eduardo | usecols     | macOS   | 48GB | Apple M4 Pro  | Y    | 12.9     | 12.8    | +2453      |
| Suryash | usecols     | windows | 32GB | Intel Core i9 | Y    | 32.3     | 32.2    | +5012      |
| Rabin   | usecols     | macOS     | 24GB | M4 Pro        | Y    | 13.0     | 12.9    | +1499      |
| Eduardo | dtype       | macOS   | 48GB | Apple M4 Pro  | Y    | 23.4     | 23.3    | -1261      |
| Suryash | dtype       | windows | 32GB | Intel Core i9 | Y    | 37.8     | 37.3    | +7066      |
| Rabin   | dtype       | macOS     | 24GB | M4 Pro        | Y    | 11.6     | 11.4    | +269       |
| Eduardo | duckdb      | macOS   | 48GB | Apple M4 Pro  | Y    | 1.2     | 10.1   | +3       |
| Rabin   | duckdb      | macOS     | 16GB | M4 pro        | Y    | 1.2     | 11.2   | +149   |